In [ ]:
import random

# --- Βοηθητικές Συναρτήσεις από Εργαστήριο 3 ---
def extended_gcd(a, b):
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def mod_inverse(a, m):
    gcd, x, y = extended_gcd(a, m)
    if gcd != 1:
        raise Exception('Δεν υπάρχει Modulo Inverse')
    return x % m

# --- Μέρος Α: Παραγωγή Μεριδίων ---
def generate_shares(secret, n, k, p):
    # Συντελεστές πολυωνύμου: Ο πρώτος (σταθερός) είναι το μυστικό.
    # Οι υπόλοιποι (k-1) είναι τυχαίοι αριθμοί.
    coefficients = [secret] + [random.randint(1, p - 1) for _ in range(k - 1)]

    shares = []
    for x in range(1, n + 1):
        # Υπολογισμός f(x)
        y = 0
        for power, coeff in enumerate(coefficients):
            # y = (y + coeff * x^power) mod p
            y = (y + coeff * (x ** power)) % p

        shares.append((x, y))

    return shares

# --- Μέρος Β: Ανακατασκευή Μυστικού ---
def reconstruct_secret(shares, p):
    secret = 0

    for i in range(len(shares)):
        xi, yi = shares[i]
        num = 1
        den = 1

        for j in range(len(shares)):
            if i != j:
                xj, _ = shares[j]
                num = (num * xj) % p
                den = (den * (xj - xi)) % p

        den_inv = mod_inverse(den, p)
        term = (yi * num * den_inv) % p
        secret = (secret + term) % p

    return secret


# Δεδομένα
SECRET_CODE = 1234
N_DIRECTORS = 5
K_THRESHOLD = 3
PRIME_MODULUS = 99991 # Πρέπει να είναι Πρώτος, μεγαλύτερος από το N και το SECRET

print(f"Το μυστικό είναι: {SECRET_CODE}")
print(f"Μοιράζεται σε {N_DIRECTORS} διευθυντές. Χρειάζονται τουλάχιστον {K_THRESHOLD}.\n")

# 1. Μοίρασμα
shares = generate_shares(SECRET_CODE, N_DIRECTORS, K_THRESHOLD, PRIME_MODULUS)
print("Τα μερίδια (x, y) που μοιράστηκαν είναι:")
for share in shares:
    print(f"Διευθυντής {share[0]}: {share}")
print("\n")

# 2. Αποτυχημένη Προσπάθεια (2 Διευθυντές)
print("--- Σενάριο 1: Έρχονται ΜΟΝΟ 2 διευθυντές (Λιγότεροι από K) ---")
failed_shares = [shares[0], shares[2]] # Διευθυντής 1 και 3
wrong_secret = reconstruct_secret(failed_shares, PRIME_MODULUS)
print(f"Προσπάθεια ανακατασκευής: {wrong_secret} (ΛΑΘΟΣ! Δεν έμαθαν τίποτα)\n")

# 3. Επιτυχημένη Προσπάθεια (3 Διευθυντές)
print("--- Σενάριο 2: Έρχονται 3 διευθυντές (Ακριβώς K) ---")
success_shares = [shares[1], shares[3], shares[4]] # Διευθυντές 2, 4 και 5
correct_secret = reconstruct_secret(success_shares, PRIME_MODULUS)
print(f"Προσπάθεια ανακατασκευής: {correct_secret} (ΕΠΙΤΥΧΙΑ! Το θησαυροφυλάκιο άνοιξε)")

In [ ]:
# !pip install shamirs

In [ ]:
import shamirs

# μυστικό και οι παράμετροι
secret_code = 1234
total_directors = 5      # N (quantity)
required_directors = 3   # K (threshold)

print(f"Αρχικό Μυστικό: {secret_code}")
print(f"Θα μοιραστεί σε {total_directors} άτομα. Απαιτούνται {required_directors}.\n")

# Παραγωγή Μεριδίων
shares = shamirs.shares(secret_code, quantity=total_directors, threshold=required_directors)

print("Τα μερίδια δημιουργήθηκαν:")
for i, share in enumerate(shares):
    # Η βιβλιοθήκη επιστρέφει αντικείμενα τύπου 'Share'.
    # Περιέχουν το x (index) και το y (value), συν τον πρώτο αριθμό που επιλέχθηκε αυτόματα
    print(f"Διευθυντής {i+1}: Μερίδιο -> {share}")

print("\n--- Ώρα να ανοίξουμε το θησαυροφυλάκιο ---")

# Σενάριο Αποτυχίας: Έρχονται μόνο 2 διευθυντές (Π.χ. ο 1ος και ο 2ος)
failed_attempt_shares = [shares[0], shares[1]]

try:
    wrong_secret = shamirs.interpolate(failed_attempt_shares)
    # Θα βγάλει ένα εντελώς άκυρο/τυχαίο νούμερο, αποδεικνύοντας τη θεωρία!
    print(f"Αποτυχία (2 άτομα): Βγήκε λάθος κωδικός -> {wrong_secret}")
except Exception as e:
    print(f"Αποτυχία (2 άτομα): Σφάλμα κατά την ανακατασκευή! ({e})")

# Σενάριο Επιτυχίας: Έρχονται 3 διευθυντές (Π.χ. ο 1ος, ο 3ος και ο 5ος)
successful_attempt_shares = [shares[0], shares[2], shares[4]]

# Η interpolate() θα ενώσει σωστά και θα βρει το f(0)
correct_secret = shamirs.interpolate(successful_attempt_shares)
print(f"\nΕπιτυχία (3 άτομα): Το μυστικό ανακατασκευάστηκε! Ο κωδικός είναι -> {correct_secret}")